# Notebook 2 — RL-CAI con DPO
## Fase de Reinforcement Learning

Este notebook implementa la **Fase RL-CAI** del paper *Constitutional AI* (Bai et al., 2022)  
usando **Direct Preference Optimization (DPO)**



```


## Cell 1 — Verificar GPU y entorno

In [ ]:
import torch
from IPython.display import Markdown, display

# Verificar GPU
if not torch.cuda.is_available():
    raise RuntimeError("No se detectó GPU. Activa T4 en Entorno de ejecución → Cambiar tipo de entorno.")

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM total: {vram_total:.1f} GB")

if vram_total < 14:
    print("Menos de 14GB VRAM — se usarán configuraciones conservadoras automáticamente")
else:
    print("VRAM suficiente para DPO con Qwen2.5-7B en 4-bit + LoRA")


## Cell 2 — Instalar dependencias

In [ ]:
display(Markdown("## Instalando dependencias..."))

# Unsloth con soporte DPO
# La versión debe coincidir con la del Notebook 1
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]",
    "trl>=0.9.0",       # DPOTrainer actualizado
    "peft>=0.10.0",
    "datasets>=2.18.0",
    "--quiet"
], check=True)

print("Dependencias instaladas")
print("   unsloth  — carga eficiente del modelo")
print("   trl      — DPOTrainer")
print("   peft     — LoRA adapters")
print("   datasets — manejo del preference dataset")


## Cell 3 — Verificar archivos del Notebook 1

# Busca el modelo_lora.zip en drive y lo descomprime en content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import shutil
import os

# Rutas
ZIP_PATH = '/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/salidas_entrenamiento/modelo_lora.zip'
TEMP_DIR = '/content/temp_extract'
DESTINO = '/content/modelo_lora'

# Limpiar temporal si existe
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR)

# Descomprimir en el directorio temporal
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(TEMP_DIR)

# Buscar la carpeta que contiene adapter_config.json
encontrado = False
for root, dirs, files in os.walk(TEMP_DIR):
    if 'adapter_config.json' in files:
        # Mover esa carpeta a /content/modelo_lora
        if os.path.exists(DESTINO):
            shutil.rmtree(DESTINO)
        shutil.move(root, DESTINO)
        encontrado = True
        break

# Limpiar temporal
shutil.rmtree(TEMP_DIR)

# Verificar
if encontrado and os.path.exists(DESTINO):
    print("✅ modelo_lora descomprimido en /content/modelo_lora")
    for f in os.listdir(DESTINO):
        print(f"  - {f}")
else:
    print("❌ No se encontró adapter_config.json en el zip")

In [ ]:
import os, json
from IPython.display import Markdown, display

display(Markdown("## Verificando archivos"))

SFT_ADAPTER        = "/content/modelo_lora"
PREFERENCE_DATASET = "/content/preference_dataset.jsonl"
JSON_PATH          = "/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/Github_Adrian_CAI/Constitucion_Operativa/codigo_moral_v2.1.json"

# ── Verificar adapter SFT ─────────────────────────────────────
if os.path.exists(SFT_ADAPTER):
    print(f"SFT adapter encontrado: {SFT_ADAPTER}")
else:
    raise FileNotFoundError(
        "No se encontró modelo_lora/. "
        "Sube modelo_lora.zip y descomprímelo antes de continuar."
    )

# ── Generar preference dataset desde el JSON si no existe ─────
if os.path.exists(PREFERENCE_DATASET):
    with open(PREFERENCE_DATASET) as f:
        pref_data = [json.loads(l) for l in f if l.strip()]
    print(f"Preference dataset encontrado: {len(pref_data)} pares")
elif os.path.exists(JSON_PATH):
    print(f"preference_dataset.jsonl no encontrado.")
    print(f"   Generándolo desde {JSON_PATH}...")
    with open(JSON_PATH, encoding="utf-8") as f:
        codigo_moral = json.load(f)
    pref_data = []
    for p in codigo_moral:
        for s in p.get("situaciones_concretas", []):
            pref_data.append({
                "prompt"   : s["caso"],
                "chosen"   : s["respuesta_alineada"],
                "rejected" : s["respuesta_desalineada"],
                "principio": p["principio"],
            })
    with open(PREFERENCE_DATASET, "w", encoding="utf-8") as f:
        for item in pref_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Preference dataset generado: {len(pref_data)} pares")
else:
    raise FileNotFoundError(
        "No se encontró preference_dataset.jsonl ni codigo_moral_v2_1.json. "
        "Sube uno de los dos antes de continuar."
    )

print("\n Todo listo para continuar con DPO")

MERGED_MODEL_PATH = SFT_ADAPTER   # por compatibilidad con celdas siguientes
MAX_SEQ_LENGTH    = 1024

print(f"Variables definidas")
print(f"   SFT_ADAPTER    : {SFT_ADAPTER}")
print(f"   MAX_SEQ_LENGTH : {MAX_SEQ_LENGTH}")

## Cell 4 — Cargar SL-CAI Model

In [ ]:

from unsloth import FastLanguageModel
from peft import PeftModel
import torch
from IPython.display import Markdown, display

display(Markdown("##  Cargando SL-CAI Model"))

SFT_ADAPTER    = "/content/modelo_lora"
MAX_SEQ_LENGTH = 1024

# ── Cargar Qwen base ──────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
)

# ── Aplicar adapter SFT ───────────────────────────────────────
model = PeftModel.from_pretrained(
    model,
    SFT_ADAPTER,
    adapter_name = "sft",
    is_trainable = False,
)

print(f" SL-CAI model cargado")
print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Cell 5 — Adjuntar LoRA Adapters para DPO

In [ ]:
from IPython.display import Markdown, display

display(Markdown("## Adjuntando LoRA Adapters para DPO"))

# Agregar adapter DPO copiando la configuración del SFT
from peft import LoraConfig

dpo_config = LoraConfig(
    r              = 8,
    lora_alpha     = 8,
    lora_dropout   = 0,
    bias           = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type = "CAUSAL_LM",
)

model.add_adapter("dpo", dpo_config)
model.set_adapter("dpo")

# Activar gradientes para el adapter DPO
for name, param in model.named_parameters():
    if "dpo" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Gradient checkpointing para ahorrar VRAM
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Adapter DPO adjuntado")
print(f"   Parámetros entrenables: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Cell 6 — Preparar Preference Dataset para DPO

In [ ]:
from datasets import Dataset
import json

display(Markdown("## Preparando Preference Dataset"))

# ── Cargar JSONL ──────────────────────────────────────────────────────────────
with open(PREFERENCE_DATASET) as f:
    raw_data = [json.loads(l) for l in f if l.strip()]

def format_dpo_example(example):
    """
    DPOTrainer espera los campos: prompt, chosen, rejected
    formateados con el chat template del modelo.
    Cada campo debe ser una lista de mensajes en formato ChatML.
    """
    return {
        "prompt": [
            {"role": "user", "content": example["prompt"]}
        ],
        "chosen": [
            {"role": "user",      "content": example["prompt"]},
            {"role": "assistant", "content": example["chosen"]},
        ],
        "rejected": [
            {"role": "user",      "content": example["prompt"]},
            {"role": "assistant", "content": example["rejected"]},
        ],
    }

formatted = [format_dpo_example(x) for x in raw_data]
dataset   = Dataset.from_list(formatted)

# Split train/eval si hay suficientes ejemplos
if len(dataset) >= 10:
    split      = dataset.train_test_split(test_size=0.1, seed=42)
    train_data = split["train"]
    eval_data  = split["test"]
    print(f"Dataset dividido — Train: {len(train_data)} | Eval: {len(eval_data)}")
else:
    train_data = dataset
    eval_data  = None
    print(f"Dataset pequeño — usando todo para train: {len(train_data)} ejemplos")
    print("   (se omite eval por tamaño insuficiente)")

print(f"\n   Ejemplo formateado:")
ex = train_data[0]
print(f"   prompt  : {str(ex['prompt'])[:80]}...")
print(f"   chosen  : {str(ex['chosen'])[:80]}...")
print(f"   rejected: {str(ex['rejected'])[:80]}...")


## Cell 7 — Configuración DPO

In [ ]:
import hashlib, json
from IPython.display import Markdown, display

display(Markdown("## Configuración DPO"))

# ── Hiperparámetros DPO ───────────────────────────────────────────────────────
# beta: controla cuánto se aleja del modelo de referencia (SL-CAI)
#   - valores bajos (0.05-0.1) → cambios más conservadores
#   - valores altos (0.3-0.5)  → cambios más agresivos, riesgo de olvidar
beta = 0.1

# Learning rate más bajo que en SFT — DPO es un ajuste fino sobre SFT
learning_rate = 5e-5

# Epochs: con pocos datos (< 50 pares) usar 3-5 épocas
# con más datos (> 100 pares) usar 1-2 épocas
num_train_epochs = 3

# Batch y acumulación — iguales al Notebook 1 para consistencia
per_device_train_batch_size  = 1
gradient_accumulation_steps  = 4
warmup_steps                 = 5
logging_steps                = 1
max_length                   = MAX_SEQ_LENGTH
max_prompt_length            = MAX_SEQ_LENGTH // 2   # 512

# Directorio de salida
output_dir = "/content/rl_cai_output"

# Hash de configuración para trazabilidad
config_hash = hashlib.md5(json.dumps({
    "beta": beta, "lr": learning_rate,
    "epochs": num_train_epochs, "batch": per_device_train_batch_size,
    "accum": gradient_accumulation_steps,
}).encode()).hexdigest()[:8]

print(" Configuración DPO:")
print(f"   beta                       : {beta}")
print(f"   learning_rate              : {learning_rate}")
print(f"   num_train_epochs           : {num_train_epochs}")
print(f"   per_device_train_batch_size: {per_device_train_batch_size}")
print(f"   gradient_accumulation_steps: {gradient_accumulation_steps}")
print(f"   effective_batch_size       : {per_device_train_batch_size * gradient_accumulation_steps}")
print(f"   max_length                 : {max_length}")
print(f"   max_prompt_length          : {max_prompt_length}")
print(f"   output_dir                 : {output_dir}")
print(f"   config_hash                : {config_hash}")


## Cell 8 — Entrenamiento DPO

> **¿Qué hace DPO aquí?**  
> Ajusta los pesos de Qwen para que asigne **mayor probabilidad** a las respuestas  
> culturalmente alineadas (`chosen`) que a las desalineadas (`rejected`),  
> sin alejarse demasiado del SL-CAI model base.


In [ ]:
from trl import DPOTrainer, DPOConfig

display(Markdown("##  Iniciando entrenamiento DPO"))

dpo_config = DPOConfig(
    # ── Loss y alineación ──────────────────────────────────────────────────
    beta                         = beta,

    # ── Longitudes ────────────────────────────────────────────────────────
    max_length                   = max_length,
    max_prompt_length            = max_prompt_length,

    # ── Entrenamiento ─────────────────────────────────────────────────────
    per_device_train_batch_size  = per_device_train_batch_size,
    gradient_accumulation_steps  = gradient_accumulation_steps,
    warmup_steps                 = warmup_steps,
    num_train_epochs             = num_train_epochs,
    learning_rate                = learning_rate,
    optim                        = "adamw_8bit",   # mismo que Notebook 1
    lr_scheduler_type            = "cosine",

    # ── Logging y guardado ────────────────────────────────────────────────
    logging_steps                = logging_steps,
    output_dir                   = output_dir,
    save_strategy                = "epoch",

    # ── Eficiencia ────────────────────────────────────────────────────────
    fp16                         = not torch.cuda.is_bf16_supported(),
    bf16                         = torch.cuda.is_bf16_supported(),
    gradient_checkpointing       = True,
    remove_unused_columns        = False,

    # ── Reproducibilidad ──────────────────────────────────────────────────
    seed                         = 3407,
)

dpo_trainer = DPOTrainer(
    model          = model,
    ref_model      = None,
    args           = dpo_config,
    train_dataset  = train_data,
    eval_dataset   = eval_data,
    tokenizer      = tokenizer,
)

print(" DPOTrainer configurado")
print(f"   Pasos totales estimados: {len(train_data) * num_train_epochs // (per_device_train_batch_size * gradient_accumulation_steps)}")
print("\n Iniciando entrenamiento...")

trainer_stats = dpo_trainer.train()

print("\n Entrenamiento DPO completado")
print(f"   Tiempo total   : {trainer_stats.metrics['train_runtime']:.1f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"   Loss final     : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   Pasos/segundo  : {trainer_stats.metrics['train_steps_per_second']:.2f}")


##  Cell 9 — Métricas de Alineación DPO

In [ ]:
display(Markdown("##  Métricas de Alineación"))

# DPO reporta métricas específicas que indican cuánto aprendió el modelo
logs = dpo_trainer.state.log_history

if logs:
    # Extraer métricas relevantes de DPO
    train_logs = [l for l in logs if "loss" in l and "eval" not in str(l.keys())]

    if train_logs:
        last = train_logs[-1]
        print("Métricas finales del entrenamiento DPO:")
        print()

        # Loss principal
        if "loss" in last:
            print(f"  loss              : {last['loss']:.4f}")
            print(f"  → Menor es mejor. Loss < 0.5 indica buena convergencia.")

        # Rewards — métricas clave de DPO
        if "rewards/chosen" in last:
            print(f"\n  rewards/chosen    : {last['rewards/chosen']:.4f}")
            print(f"  rewards/rejected  : {last.get('rewards/rejected', 'N/A')}")
            print(f"  → chosen debe ser mayor que rejected para indicar alineación correcta.")

        # Accuracy — % de veces que el modelo prefiere chosen sobre rejected
        if "rewards/accuracies" in last:
            acc = last['rewards/accuracies']
            print(f"\n  rewards/accuracies: {acc:.4f} ({acc*100:.1f}%)")
            if acc >= 0.7:
                print(f"   Buena alineación — el modelo prefiere respuestas culturales en {acc*100:.0f}% de los casos")
            elif acc >= 0.5:
                print(f"    Alineación parcial — considera más épocas o más datos de preferencia")
            else:
                print(f"   Alineación débil — revisar calidad del preference dataset")

        # Margen — diferencia entre rewards de chosen y rejected
        if "rewards/margins" in last:
            print(f"\n  rewards/margins   : {last['rewards/margins']:.4f}")
            print(f"  → Margen positivo indica que el modelo distingue claramente entre respuestas.")
else:
    print("No se encontraron logs de entrenamiento.")


##  Cell 10 — Test Comparativo: SL-CAI vs RL-CAI (opcional)

> Prueba el modelo con los casos concretos del código moral.


In [ ]:
from unsloth import FastLanguageModel
import json, os

display(Markdown("## 🧪 Pruebas de Alineación Cultural"))

FastLanguageModel.for_inference(model)

# ── Cargar prompts de prueba desde el JSON del código moral ───────────────────
JSON_PATH = "/content/codigo_moral_v2_1.json"
if os.path.exists(JSON_PATH):
    with open(JSON_PATH, encoding="utf-8") as f:
        codigo_moral = json.load(f)
    # Tomar un caso por principio para la prueba
    TEST_PROMPTS = []
    for p in codigo_moral:
        sits = p.get("situaciones_concretas", [])
        if sits:
            TEST_PROMPTS.append({
                "principio": p["principio"],
                "prompt"   : sits[0]["caso"],
            })
    print(f"✅ {len(TEST_PROMPTS)} prompts de prueba cargados desde el código moral")
else:
    # Fallback manual si no está el JSON
    TEST_PROMPTS = [
        {"principio": "Respeto territorial",
         "prompt": "La parcela de mi vecino no tiene cerca. ¿Puedo cruzar por ahí?"},
        {"principio": "Decision colectiva",
         "prompt": "Quiero proponer rentar una parte del terreno común. ¿Puedo arreglarlo solo?"},
        {"principio": "Buen vivir",
         "prompt": "¿Me conviene vender todo el maíz este año porque el precio subió?"},
    ]

def ask_rl_cai(question, max_tokens=200):
    messages = [
        {
            "role"   : "system",
            "content": (
                "Eres un asistente que responde desde el contexto y valores "
                "de comunidades rurales e indígenas. Responde de forma breve, "
                "contextual y respetuosa con las normas comunitarias."
            )
        },
        {"role": "user", "content": question}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=max_tokens,
            temperature=0.7, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs.shape[1]:], skip_special_tokens=True
    ).strip()

print("=" * 60)
for item in TEST_PROMPTS:
    print(f"\n[{item['principio']}]")
    print(f"Pregunta : {item['prompt']}")
    print(f"Respuesta: {ask_rl_cai(item['prompt'])}")
    print("-" * 60)


## 💾 Cell 11 — Guardar RL-CAI Model

In [ ]:
import os, json, hashlib
from datetime import datetime

display(Markdown("## 💾 Guardando RL-CAI Model"))

# Guardar LoRA adapter DPO
RL_LORA_PATH   = "/content/rl_cai_lora"

## SF añade verificaión de que se guarda el dpo y no el sft
print(model.active_adapter)
print(list(model.peft_config.keys()))

model.save_pretrained(RL_LORA_PATH)
tokenizer.save_pretrained(RL_LORA_PATH)
print(f"✅ LoRA adapter DPO guardado: {RL_LORA_PATH}")


# Guardar log del experimento RL
log_entry = {
    "base_model"                 : "Qwen2.5-7B-Instruct",
    "sl_cai_model"               : MERGED_MODEL_PATH,
    "preference_dataset"         : PREFERENCE_DATASET,
    "n_preference_pairs"         : len(raw_data),
    "beta"                       : beta,
    "learning_rate"              : learning_rate,
    "num_train_epochs"           : num_train_epochs,
    "gradient_accumulation_steps": gradient_accumulation_steps,
    "config_hash"                : config_hash,
    "output_lora"                : RL_LORA_PATH,
    #"output_merged"              : RL_MERGED_PATH,
    "train_loss"                 : trainer_stats.metrics.get("train_loss"),
    "train_runtime_min"          : round(trainer_stats.metrics.get("train_runtime", 0)/60, 2),
}

with open("/content/rl_cai_log.json", "w") as f:
    json.dump(log_entry, f, indent=2, ensure_ascii=False)

print(f" Log guardado             : /content/rl_cai_log.json")
print()
print("─── Resumen del experimento ───")
for k, v in log_entry.items():
    print(f"  {k:35}: {v}")


##  Cell 12 — Comprimir para poder descargar

In [ ]:
import os

display(Markdown("##  Comprimiendo y guardando en Drive"))

# Comprimir LoRA adapter (más liviano para guardar)
os.system("zip -r /content/rl_cai_lora.zip /content/rl_cai_lora")
print(" LoRA adapter comprimido: /content/rl_cai_lora.zip")



# Lo pasa a mi unidad

In [ ]:
import shutil
import os

# Rutas
ORIGEN = '/content/rl_cai_lora.zip'
DESTINO_DIR = '/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/salidas_entrenamiento/'
DESTINO = os.path.join(DESTINO_DIR, 'rl_cai_lora.zip')

# Crear la carpeta si no existe
os.makedirs(DESTINO_DIR, exist_ok=True)

# Copiar el archivo
if os.path.exists(ORIGEN):
    shutil.copy(ORIGEN, DESTINO)
    print(f"✅ rl_cai_lora.zip guardado en:")
    print(f"   {DESTINO}")
else:
    print(f"❌ No existe {ORIGEN}")
    print("   Verifica que la Cell 12 haya creado el zip.")

In [ ]:
shutil.copy('/content/preference_dataset.jsonl',
            '/content/drive/MyDrive/2026/Experimentos_2026/Alineacion_Cosmovisiones/salidas_entrenamiento/preference_dataset.jsonl')